# Packages

In [1]:
# Basic Package Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Non-basic package imports
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.nn import CrossEntropyLoss
import requests
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

# Packages I don't understand
from fcd_torch import FCD
import rdkit
from collections import Counter
import gc
import pickle
import wandb

# Add the Python_files directory to the Python path
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'Python_files'))

# Now you can import your modules
import functions_enc as f
import function_depot as fd

# Functions

## Means and Standard Deviations Confusion Matrices

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.metrics import confusion_matrix
import os
from glob import glob
def create_percentage_confusion_matrix_mean_std(folder, 
                                                folder_name, 
                                                bin_size, 
                                                threshold_size, use_super_test=False, 
                                                save_plots=True, 
                                                test_in=True,
                                                train_in=False, 
                                                num_loops=None):
    """
    Compute mean and stddev of percentage confusion matrices over multiple looped output files.
    Will process multiple parquet files matching the base name pattern (before loop number).
    Plots the mean confusion matrix; each cell annotated with "mean ± std".
    Optionally limit the number of loops/files used with num_loops.
    """

    if not test_in and not train_in:
        print("No data to use, toggle train_in or test_in to True.")
        return

    def response_to_tox_class_local(response_value):
        if response_value <= 5:
            return 0
        elif response_value <= 50:
            return 1
        elif response_value <= 500:
            return 2
        elif response_value <= 5000:
            return 3
        else:
            return 4

    # Prepare the file pattern for glob
    folder_path = os.path.join(folder, folder_name)
    bin_part = str(bin_size).replace('.', '_')
    threshold_part = str(threshold_size).replace('.', '_')
    base_fp = (
        f"super_test_cond_enc_bin{bin_part}_thresh{threshold_part}_df_spectra"
        if use_super_test else f"cond_enc_bin{bin_part}_thresh{threshold_part}_df_spectra"
    )
    file_pattern = os.path.join(folder_path, f"{base_fp}_loop*.parquet")

    file_list = sorted(glob(file_pattern))
    if len(file_list) == 0:
        raise FileNotFoundError(f"No files found matching pattern: {file_pattern}")
    if num_loops is not None:
        file_list = file_list[:num_loops]

    all_cm_percent = []
    n_classes = 5
    class_labels = ["Tox Level 0", "Tox Level 1", "Tox Level 2", "Tox Level 3", "Tox Level 4"]

    for fname in file_list:
        df = pd.read_parquet(fname)
        # Filter based on train_in and test_in flags
        if 'train' in df.columns:
            if train_in and test_in:
                # Include both train and test
                pass
            elif train_in:
                # Include only train
                df = df[df['train'] == 1].copy()
            elif test_in:
                # Include only test
                df = df[df['train'] == 0].copy()
        
        df['tox_class'] = df['Response'].apply(response_to_tox_class_local)

        y_true = df['tox_class'].values
        y_pred = df['cond_tox_pred_class'].values

        cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
        # Row-wise normalization (%)
        with np.errstate(divide='ignore', invalid='ignore'):
            cm_percent = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
            cm_percent = np.nan_to_num(cm_percent)
        all_cm_percent.append(cm_percent)

    all_cm_percent = np.array(all_cm_percent)  # shape (n_files, n_classes, n_classes)
    mean_cm = np.mean(all_cm_percent, axis=0)
    std_cm = np.std(all_cm_percent, axis=0)

    # Plotting
    plt.figure(figsize=(10, 8))
    annot = np.empty_like(mean_cm, dtype=object)
    for i in range(n_classes):
        for j in range(n_classes):
            annot[i, j] = f"{mean_cm[i, j]:.1f}±{std_cm[i, j]:.1f}"

    sns.heatmap(
        mean_cm, annot=annot, fmt='', cmap='Blues',
        xticklabels=class_labels, yticklabels=class_labels,
        cbar_kws={'label': 'Mean Percentage (%)'},
        square=True
    )

    test_type = "Super Test" if use_super_test else "Regular Test"
    # Build data label based on train_in and test_in
    if train_in and test_in:
        data_label = " (Train + Test)"
    elif train_in:
        data_label = " (Train Only)"
    elif test_in:
        data_label = " (Test Only)"
    else:
        data_label = ""
    
    plt.title(f"Confusion Matrix (Mean±Std%) - {test_type}{data_label} (Bin={bin_size}, Threshold={threshold_size})\nn={len(file_list)} loops")
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()

    if save_plots:
        save_path = os.path.join(folder_path, f"confusion_matrix_percentages_meanstd_{'super_test_' if use_super_test else ''}bin{bin_size}_thr{threshold_size}_loops.png")
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved as: {save_path}")

    plt.show()

## Direct Prediction Model Confusion Matrix

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import os
from glob import glob

def create_confusion_matrix_general_mean_std(
    folder_path,
    bin_val,
    threshold_val,
    pred_class_col='cond_tox_pred_class',
    response_col='Response',
    train_col='train',
    file_prefix='cond_enc',
    use_super_test=False,
    test_in=True,
    train_in=False,
    save_plots=False,
    figsize=(10, 8),
    num_loops=None  # New argument to limit number of loops/files
):
    if not test_in and not train_in:
        print("No data to use, toggle train_in or test_in to True.")
        return

    bin_part = str(bin_val).replace('.', '_')
    threshold_part = str(threshold_val).replace('.', '_')
    if use_super_test:
        base_fp = f"super_test_{file_prefix}_bin{bin_part}_thresh{threshold_part}_df_spectra"
    else:
        base_fp = f"{file_prefix}_bin{bin_part}_thresh{threshold_part}_df_spectra"
    file_pattern = os.path.join(folder_path, f"{base_fp}_loop*.parquet")
    file_list = sorted(glob(file_pattern))
    if not file_list:
        raise FileNotFoundError(f"No files found matching pattern: {file_pattern}")
    if num_loops is not None:
        file_list = file_list[:num_loops]
    n_classes = 5
    class_labels = ["Tox Level 0", "Tox Level 1", "Tox Level 2", "Tox Level 3", "Tox Level 4"]

    all_cm_percent = []

    for fname in file_list:
        df = pd.read_parquet(fname)
        if train_col in df.columns:
            # Filter based on train_in and test_in flags
            if train_in and test_in:
                # Include both train and test
                pass
            elif train_in:
                # Include only train
                df = df[df[train_col] == 1].copy()
            elif test_in:
                # Include only test
                df = df[df[train_col] == 0].copy()
        
        def response_to_tox_class(response_value):
            if response_value <= 5:
                return 0
            elif response_value <= 50:
                return 1
            elif response_value <= 500:
                return 2
            elif response_value <= 5000:
                return 3
            else:
                return 4
        df['tox_class'] = df[response_col].apply(response_to_tox_class)
        y_true = df['tox_class'].values
        y_pred = df[pred_class_col].values
        cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
        with np.errstate(divide='ignore', invalid='ignore'):
            cm_percent = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
            cm_percent = np.nan_to_num(cm_percent)
        all_cm_percent.append(cm_percent)

    all_cm_percent = np.array(all_cm_percent)
    mean_cm = np.mean(all_cm_percent, axis=0)
    std_cm = np.std(all_cm_percent, axis=0)

    plt.figure(figsize=figsize)
    annot = np.empty_like(mean_cm, dtype=object)
    for i in range(n_classes):
        for j in range(n_classes):
            annot[i, j] = f"{mean_cm[i, j]:.1f}±{std_cm[i, j]:.1f}"

    sns.heatmap(
        mean_cm, annot=annot, fmt='', cmap='Blues',
        xticklabels=class_labels, yticklabels=class_labels,
        cbar_kws={'label': 'Mean Percentage (%)'},
        square=True
    )

    test_type = "Super Test" if use_super_test else "Regular Test"
    # Build data label based on train_in and test_in
    if train_in and test_in:
        data_label = " (Train + Test)"
    elif train_in:
        data_label = " (Train Only)"
    elif test_in:
        data_label = " (Test Only)"
    else:
        data_label = ""
    
    plt.title(f"Confusion Matrix (Mean±Std%) - {test_type}{data_label}\n(Bin={bin_val}, Threshold={threshold_val})\nn={len(file_list)} loops")
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()

    if save_plots:
        save_path = os.path.join(folder_path, f"confusion_matrix_percentages_meanstd_{'super_test_' if use_super_test else ''}bin{bin_val}_thr{threshold_val}_loops.png")
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved as: {save_path}")

    plt.show()

# Visualization Executions

## 2 Step Algorithm Visualizations

## Direct Algorithm Visualizations

## Random Forest Classifier Visualizations